# Lab 3: Agentes - O que são e como criar (30 min)

## Objetivos
- Entender o que é um Agente no Foundry
- Criar um agente básico com o SDK
- Adicionar Tools (Code Interpreter, File Search)
- Ver como publicar no M365 Copilot

## 📖 O que é um Agente?

Um **Agente** é um assistente de IA que:
- Tem **instruções** (system prompt) que definem o seu comportamento
- Pode usar **ferramentas (tools)** para executar ações
- Mantém **conversas** com histórico (threads)
- Pode aceder a **ficheiros e dados** (knowledge)

### Diferença: Modelo vs Agente

```
Modelo (GPT-4o)          Agente
┌──────────────┐        ┌──────────────────────────┐
│ Input → Output│       │ Instruções + Personalidade│
│ (stateless)   │       │ + Tools (código, search)  │
│               │       │ + Knowledge (ficheiros)   │
│               │       │ + Threads (memória)       │
└──────────────┘        └──────────────────────────┘
```

### Tools Disponíveis

| Tool | Descrição |
|------|-----------|
| **Code Interpreter** | Executa Python (cálculos, gráficos, análise) |
| **File Search** | Pesquisa em ficheiros carregados (RAG) |
| **Bing Grounding** | Pesquisa na web em tempo real |
| **Azure AI Search** | Pesquisa em índices do AI Search |
| **Function Calling** | Chama as tuas próprias APIs/funções |
| **OpenAPI** | Integra APIs externas via spec OpenAPI |

In [ ]:
# Setup
from dotenv import load_dotenv
import os

load_dotenv("../.env")

from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential
from azure.core.credentials import AzureKeyCredential

# Ligar ao projeto Foundry
project_client = AIProjectClient(
    endpoint=os.getenv("AZURE_AI_FOUNDRY_ENDPOINT"),
    credential=AzureKeyCredential(os.getenv("AZURE_AI_FOUNDRY_KEY")),
)

print("✅ Ligado ao Azure AI Foundry!")

## 💻 Exercício 3.1: Criar um Agente Básico

Vamos criar um agente simples que é um assistente de português.

In [ ]:
# Criar o agente
agent = project_client.agents.create_agent(
    model=os.getenv("AZURE_OPENAI_DEPLOYMENT", "gpt-4o"),
    name="Assistente Workshop",
    instructions="""
    Tu és um assistente simpático chamado 'Tónio'. 
    Respondes sempre em português de Portugal.
    Usas emojis para tornar as respostas mais divertidas.
    Quando não sabes algo, admites honestamente.
    """,
)

print(f"✅ Agente criado!")
print(f"   ID: {agent.id}")
print(f"   Nome: {agent.name}")
print(f"   Modelo: {agent.model}")

In [ ]:
# Criar uma thread (conversa) e enviar mensagem
thread = project_client.agents.threads.create()
print(f"📝 Thread criada: {thread.id}")

# Enviar mensagem
message = project_client.agents.messages.create(
    thread_id=thread.id,
    role="user",
    content="Olá Tónio! Apresenta-te e diz-me o que sabes fazer.",
)

# Executar o agente
run = project_client.agents.runs.create_and_process(
    thread_id=thread.id,
    agent_id=agent.id,
)

print(f"🏃 Run status: {run.status}")

# Ver a resposta
messages = project_client.agents.messages.list(thread_id=thread.id)
for msg in reversed(list(messages)):
    role = "👤" if msg.role == "user" else "🤖"
    for content in msg.content:
        if hasattr(content, "text"):
            print(f"{role} {content.text.value}")
    print()

In [ ]:
# Continuar a conversa (multi-turno)
message = project_client.agents.messages.create(
    thread_id=thread.id,
    role="user",
    content="Qual é a capital de Portugal e quantos habitantes tem?",
)

run = project_client.agents.runs.create_and_process(
    thread_id=thread.id,
    agent_id=agent.id,
)

# Ver apenas a última resposta
messages = project_client.agents.messages.list(thread_id=thread.id)
ultima = list(messages)[0]  # A mais recente
for content in ultima.content:
    if hasattr(content, "text"):
        print(f"🤖 {content.text.value}")

## 💻 Exercício 3.2: Agente com Code Interpreter

O **Code Interpreter** permite ao agente executar Python. Útil para cálculos, gráficos, e análise de dados.

In [ ]:
from azure.ai.projects.models import CodeInterpreterTool

# Criar agente com Code Interpreter
agente_code = project_client.agents.create_agent(
    model=os.getenv("AZURE_OPENAI_DEPLOYMENT", "gpt-4o"),
    name="Analista de Dados",
    instructions="""
    Tu és um analista de dados. Respondes em português de Portugal.
    Quando te pedem cálculos ou gráficos, usa o Code Interpreter.
    Mostra sempre o código que executaste.
    """,
    tools=CodeInterpreterTool().definitions,
)

print(f"✅ Agente com Code Interpreter criado: {agente_code.id}")

In [ ]:
# Pedir ao agente para fazer cálculos
thread2 = project_client.agents.threads.create()

project_client.agents.messages.create(
    thread_id=thread2.id,
    role="user",
    content="""
    Tenho estes dados de vendas mensais (em milhares de euros):
    Jan: 45, Fev: 52, Mar: 48, Abr: 61, Mai: 55, Jun: 70
    
    Calcula a média, o mês com mais vendas, e a tendência de crescimento.
    """,
)

run2 = project_client.agents.runs.create_and_process(
    thread_id=thread2.id,
    agent_id=agente_code.id,
)

# Ver resposta
messages = project_client.agents.messages.list(thread_id=thread2.id)
for msg in reversed(list(messages)):
    if msg.role == "assistant":
        for content in msg.content:
            if hasattr(content, "text"):
                print(content.text.value)

## 💻 Exercício 3.3: Agente com Function Calling

Vamos criar um agente que pode chamar funções que nós definimos — por exemplo, consultar o "estado do tempo".

In [ ]:
import json
from azure.ai.projects.models import FunctionTool, ToolSet

# Definir a nossa "função" (simulada)
def obter_tempo(cidade: str) -> str:
    """Simula obter o estado do tempo para uma cidade."""
    dados_tempo = {
        "lisboa": {"temp": 22, "estado": "☀️ Soalheiro", "humidade": 45},
        "porto": {"temp": 18, "estado": "🌧️ Chuva ligeira", "humidade": 75},
        "faro": {"temp": 26, "estado": "☀️ Muito sol", "humidade": 30},
        "coimbra": {"temp": 20, "estado": "⛅ Parcialmente nublado", "humidade": 55},
    }
    resultado = dados_tempo.get(cidade.lower(), {"temp": 20, "estado": "Sem dados", "humidade": 50})
    return json.dumps(resultado, ensure_ascii=False)

# Registar a função como tool
functions = FunctionTool(
    functions=[
        {
            "name": "obter_tempo",
            "description": "Obtém o estado do tempo atual para uma cidade portuguesa",
            "parameters": {
                "type": "object",
                "properties": {
                    "cidade": {
                        "type": "string",
                        "description": "Nome da cidade (ex: Lisboa, Porto, Faro)",
                    }
                },
                "required": ["cidade"],
            },
        }
    ]
)

toolset = ToolSet()
toolset.add(functions)

# Criar agente com function calling
agente_tempo = project_client.agents.create_agent(
    model=os.getenv("AZURE_OPENAI_DEPLOYMENT", "gpt-4o"),
    name="Meteorologista",
    instructions="Tu és um meteorologista português simpático. Usa a função obter_tempo para dar informações do tempo.",
    toolset=toolset,
)

print(f"✅ Agente Meteorologista criado: {agente_tempo.id}")

In [ ]:
# Testar o agente com function calling
thread3 = project_client.agents.threads.create()

project_client.agents.messages.create(
    thread_id=thread3.id,
    role="user",
    content="Como está o tempo em Lisboa e no Porto hoje?",
)

# create_and_process com toolset faz o function calling automaticamente
run3 = project_client.agents.runs.create_and_process(
    thread_id=thread3.id,
    agent_id=agente_tempo.id,
)

print(f"🏃 Run status: {run3.status}")

# Ver resposta
messages = project_client.agents.messages.list(thread_id=thread3.id)
for msg in reversed(list(messages)):
    role = "👤" if msg.role == "user" else "🤖"
    for content in msg.content:
        if hasattr(content, "text"):
            print(f"{role} {content.text.value}")
    print()

## 📖 Publicar no M365 Copilot

Os agentes criados no Foundry podem ser publicados como **Plugins do Microsoft 365 Copilot**:

1. No portal Foundry, vai ao teu agente
2. Clica em **"Deploy"** → **"Microsoft 365 Copilot"**
3. O agente fica disponível como extensão no Copilot do Teams/Word/etc.

**Requisitos:**
- Licença Microsoft 365 Copilot
- Admin approval (IT admin)
- O agente deve estar numa região suportada

> ⚠️ No workshop não vamos publicar (requer licença M365 Copilot), mas fica a saber que é possível!

In [ ]:
# Limpeza - apagar os agentes criados neste lab
for a in [agent, agente_code, agente_tempo]:
    try:
        project_client.agents.delete_agent(a.id)
        print(f"🗑️ Agente '{a.name}' apagado")
    except Exception as e:
        print(f"⚠️ Erro ao apagar '{a.name}': {e}")

print("\n✅ Limpeza concluída!")

## ✅ Resumo

Neste lab aprendeste:
- A diferença entre um modelo e um agente
- Como criar agentes com o SDK Python
- Como usar threads para conversas multi-turno
- Como adicionar Code Interpreter para cálculos
- Como usar Function Calling para integrar com APIs
- Que é possível publicar agentes no M365 Copilot

**Próximo:** [Lab 4 - Knowledge & RAG](lab04-knowledge-rag.ipynb)